## Importing libraries

In [1]:
import pandas as pd
import numpy as np
import spacy
import re

import warnings
warnings.filterwarnings('ignore')

/opt/micromamba/lib/python3.12/site-packages/torch/cuda/__init__.py:1074: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


## Loading the dataset

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zynicide/wine-reviews")

print("Path to dataset files:", path)

Path to dataset files: /home/jupyter/.cache/kagglehub/datasets/zynicide/wine-reviews/versions/4


In [4]:
df = pd.read_csv(path + "/winemag-data-130k-v2.csv")
df.head()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            92506 non-null   str    
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  str    
 7   region_1               108724 non-null  str    
 8   region_2               50511 non-null   str    
 9   taster_name            103727 non-null  str    
 10  taster_twitter_handle  98758 non-null   str    
 11  title                  129971 non-null  str    
 12  variety                129970 non-null  str    
 13  winery                 129971 non-null  str    
dtypes: float64(1), int64(2), str(11)
memory usage: 

In [6]:
print(f"Shape: {df.shape}")
print(f"Number of Missing Values: {df.isnull().sum().sum()}")
print(f"Unique Values of the Target Variable 'state': {df['country'].unique()}")

Shape: (129971, 14)
Number of Missing Values: 204752
Unique Values of the Target Variable 'state': <ArrowStringArray>
[                 'Italy',               'Portugal',                     'US',
                  'Spain',                 'France',                'Germany',
              'Argentina',                  'Chile',              'Australia',
                'Austria',           'South Africa',            'New Zealand',
                 'Israel',                'Hungary',                 'Greece',
                'Romania',                 'Mexico',                 'Canada',
                      nan,                 'Turkey',         'Czech Republic',
               'Slovenia',             'Luxembourg',                'Croatia',
                'Georgia',                'Uruguay',                'England',
                'Lebanon',                 'Serbia',                 'Brazil',
                'Moldova',                'Morocco',                   'Peru',
             

## Text Preprocessing

In [7]:
!python -m spacy download en_core_web_sm

/opt/micromamba/lib/python3.12/site-packages/torch/cuda/__init__.py:1074: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 103.9 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [8]:
nlp = spacy.load("en_core_web_sm")

In [9]:
def clean_text_lemmatize(text):
    # Lowercase
    text = text.lower()

    # Remove punctuation and special characters
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Process with spaCy
    doc = nlp(text)

    # Lemmatize
    tokens = [
        token.lemma_
        for token in doc
        if token.lemma_.strip() != ""  # Remove empty tokens
        and len(token.lemma_) > 2      # Remove very short tokens (likely noise)
    ]

    return " ".join(tokens)

In [10]:
# Apply text cleaning
df["clean_description"] = df["description"].apply(clean_text_lemmatize)

In [17]:
df.to_csv('./input/wine_reviews_cleaned.csv', index=False)